# Man vs Machine on Oslo Bors - Evaluation

### Data setup

In [20]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Loading and aligning datasets...")

# 1. Load your Simulation Returns (from Notebook 3)
sim_returns_mod = pd.read_csv('../Data/portfolio_simulation_returns_mod.csv')
sim_returns_mod['TradeDate'] = pd.to_datetime(sim_returns_mod['TradeDate']) + pd.offsets.MonthEnd(0)

sim_returns_hist = pd.read_csv('../Data/portfolio_simulation_returns_hist.csv')
sim_returns_hist['TradeDate'] = pd.to_datetime(sim_returns_hist['TradeDate']) + pd.offsets.MonthEnd(0)

# 2. Load Ødegaard's Risk-Free Rate (Note: skiprows=1 to bypass the text header)
rf_df = pd.read_csv('../Data/Norway_Rf_monthly.csv', skiprows=1)
rf_df['TradeDate'] = pd.to_datetime(rf_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
rf_df = rf_df[['TradeDate', 'Rf(1m)']]

# 3. Load Ødegaard's Market Portfolios (We specifically want 'VW')
mkt_df = pd.read_csv('../Data/Norway_market_portfolios_monthly.csv')
mkt_df['TradeDate'] = pd.to_datetime(mkt_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
mkt_df = mkt_df[['TradeDate', 'VW']] 

# 4. Load Ødegaard's Fama-French Factors (SMB, HML, UMD)
ff_df = pd.read_csv('../Data/Norway_pricing_factors_monthly.csv')
ff_df['TradeDate'] = pd.to_datetime(ff_df['date'], format='%Y%m%d') + pd.offsets.MonthEnd(0)
ff_df = ff_df[['TradeDate', 'SMB', 'HML', 'UMD']]

# 5. Master Merge
eval_df_mod = sim_returns_mod.merge(rf_df, on='TradeDate', how='left')
eval_df_mod = eval_df_mod.merge(mkt_df, on='TradeDate', how='left')
eval_df_mod = eval_df_mod.merge(ff_df, on='TradeDate', how='left')

eval_df_hist = sim_returns_hist.merge(rf_df, on='TradeDate', how='left')
eval_df_hist = eval_df_hist.merge(mkt_df, on='TradeDate', how='left')
eval_df_hist = eval_df_hist.merge(ff_df, on='TradeDate', how='left')

# Drop any rows where Ødegaard's data might be missing at the very end of our sample
eval_df_mod = eval_df_mod.dropna().reset_index(drop=True)
eval_df_hist = eval_df_hist.dropna().reset_index(drop=True)


# 6. Calculate the Market Risk Premium (Rm - Rf)
eval_df_mod['Mkt-RF'] = eval_df_mod['VW'] - eval_df_mod['Rf(1m)']
eval_df_hist['Mkt-RF'] = eval_df_hist['VW'] - eval_df_hist['Rf(1m)']

# 6.5 add momentum column to the historical dataframe
eval_df_hist['Net_Ret_Mom_LS'] = eval_df_mod['Net_Ret_Mom_LS']

# 7. Define the specific strategies we want to evaluate
# This list will make looping through the risk metrics and regressions much cleaner!
strategy_cols_mod = [
    'Net_Ret_LongOnly',       # The Mutual Fund AI strategy
    'Net_Ret_LongShort',      # The Hedge Fund AI strategy (Ensemble)
    'Net_Ret_Modern_XGB_LS',  # The XGBoost Hedge Fund strategy
    'Net_Ret_Mom_LS',         # The Traditional Finance baseline
    'VW'                      # The overall Market
]

strategy_cols_hist = [
    'Net_Ret_LongOnly',       # The Mutual Fund AI strategy
    'Net_Ret_LongShort',      # The Hedge Fund AI strategy (Ensemble)
    'Net_Ret_Hist_XGB_LS',    # The XGBoost Hedge Fund strategy
    'Net_Ret_Mom_LS',         # The Traditional Finance baseline
    'VW'                      # The overall Market
]

print(f"Data successfully aligned! Total valid months: {eval_df_mod.shape[0]}")
print("\nPreview of the Evaluation Matrix:")
display(eval_df_mod[['TradeDate'] + strategy_cols_mod + ['Rf(1m)', 'Mkt-RF', 'SMB', 'HML', 'UMD']].head())

Loading and aligning datasets...
Data successfully aligned! Total valid months: 35

Preview of the Evaluation Matrix:


,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Modern_XGB_LS,Net_Ret_Mom_LS,VW,Rf(1m),Mkt-RF,SMB,HML,UMD
0,2022-01-31,0.011158,0.069506,0.056486,0.084627,-0.020529,0.00073,-0.021259,-0.029103,0.107942,0.027788
1,2022-02-28,0.023632,-0.070311,-0.042289,-0.028051,0.028558,0.00074,0.027818,-0.072052,0.152077,0.065305
2,2022-03-31,0.009737,0.082550,0.051530,0.009001,0.065944,0.00087,0.065074,0.038270,0.077338,0.063125
3,2022-04-30,0.070674,0.034709,0.078051,0.045937,-0.018648,0.00081,-0.019458,-0.040243,0.096603,0.009725
4,2022-05-31,-0.057276,0.028220,0.007939,-0.003264,0.052633,0.00083,0.051803,0.078451,0.034903,0.068843


In [21]:
display(eval_df_hist)
display(eval_df_mod)

,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Benchmark,Turnover_LongOnly,Turnover_LongShort,Net_Ret_Hist_XGB_LS,Rf(1m),VW,SMB,HML,UMD,Mkt-RF,Net_Ret_Mom_LS
0,2022-01-31,-0.003205,0.052164,-0.021959,1.000000,2.000000,0.052955,0.00073,-0.020529,-0.029103,0.107942,0.027788,-0.021259,0.084627
1,2022-02-28,0.075561,-0.011558,0.073510,0.923077,1.538462,-0.011254,0.00074,0.028558,-0.072052,0.152077,0.065305,0.027818,-0.028051
2,2022-03-31,0.018616,0.106663,-0.019149,0.830189,1.358491,0.104928,0.00087,0.065944,0.038270,0.077338,0.063125,0.065074,0.009001
3,2022-04-30,0.103755,0.078269,0.052168,0.925926,1.555556,0.102938,0.00081,-0.018648,-0.040243,0.096603,0.009725,-0.019458,0.045937
4,2022-05-31,-0.044353,0.041944,-0.066441,0.750000,1.464286,0.040319,0.00083,0.052633,0.078451,0.034903,0.068843,0.051803,-0.003264
5,2022-06-30,-0.002911,0.002313,0.018326,0.571429,1.107143,-0.014416,0.00114,-0.087680,0.008334,-0.000904,0.043376,-0.088820,-0.023316
6,2022-07-31,0.053776,0.077539,0.017478,0.793103,1.344828,0.061099,0.00135,0.056899,0.006467,-0.008872,-0.025862,0.055549,0.082034
7,2022-08-31,-0.087983,0.082531,-0.125781,0.551724,1.068966,0.064210,0.00169,0.013373,0.000874,0.090788,0.102345,0.011683,0.042863
8,2022-09-30,0.080691,0.076137,0.017229,0.745763,1.457627,0.057721,0.00225,-0.107123,-0.014143,0.040670,0.050118,-0.109373,0.071689
9,2022-10-31,0.032992,0.049617,0.020553,0.745763,1.593220,0.059579,0.00228,0.091229,0.026409,0.130273,0.001536,0.088949,0.004392


,TradeDate,Net_Ret_LongOnly,Net_Ret_LongShort,Net_Ret_Benchmark,Turnover_LongOnly,Turnover_LongShort,Net_Ret_Modern_XGB_LS,Net_Ret_Mom_LS,Rf(1m),VW,SMB,HML,UMD,Mkt-RF
0,2022-01-31,0.011158,0.069506,-0.021959,1.000000,2.000000,0.056486,0.084627,0.00073,-0.020529,-0.029103,0.107942,0.027788,-0.021259
1,2022-02-28,0.023632,-0.070311,0.073510,0.807692,1.500000,-0.042289,-0.028051,0.00074,0.028558,-0.072052,0.152077,0.065305,0.027818
2,2022-03-31,0.009737,0.082550,-0.019149,0.754717,1.320755,0.051530,0.009001,0.00087,0.065944,0.038270,0.077338,0.063125,0.065074
3,2022-04-30,0.070674,0.034709,0.052168,1.000000,1.629630,0.078051,0.045937,0.00081,-0.018648,-0.040243,0.096603,0.009725,-0.019458
4,2022-05-31,-0.057276,0.028220,-0.066441,1.071429,1.750000,0.007939,-0.003264,0.00083,0.052633,0.078451,0.034903,0.068843,0.051803
5,2022-06-30,0.009415,0.002932,0.018326,1.000000,1.571429,0.016050,-0.023316,0.00114,-0.087680,0.008334,-0.000904,0.043376,-0.088820
6,2022-07-31,0.060271,0.055902,0.017478,0.931034,1.620690,0.045400,0.082034,0.00135,0.056899,0.006467,-0.008872,-0.025862,0.055549
7,2022-08-31,-0.093734,0.071825,-0.125781,0.724138,1.241379,0.055374,0.042863,0.00169,0.013373,0.000874,0.090788,0.102345,0.011683
8,2022-09-30,0.048441,0.037943,0.017229,0.813559,1.559322,0.039847,0.071689,0.00225,-0.107123,-0.014143,0.040670,0.050118,-0.109373
9,2022-10-31,0.034629,0.058709,0.020553,0.745763,1.491525,0.057362,0.004392,0.00228,0.091229,0.026409,0.130273,0.001536,0.088949


### Modern evaluation

In [22]:
# ==========================================
# CELL 2: RISK METRICS (SHARPE & DRAWDOWN)
# ==========================================
import pandas as pd
import numpy as np

print("Calculating Risk Metrics...\n")

risk_results = []

for col in strategy_cols_mod:
    # 1. Calculate Excess Return for the strategy
    # Note: If the strategy is 'VW', it is already the market, but we still subtract Rf for its Sharpe
    excess_return = eval_df_mod[col] - eval_df_mod['Rf(1m)']
    
    # 2. Calculate Annualized Return and Volatility
    ann_ret = eval_df_mod[col].mean() * 12
    ann_vol = eval_df_mod[col].std() * np.sqrt(12)
    
    # 3. Calculate Annualized Sharpe Ratio
    ann_excess_ret = excess_return.mean() * 12
    sharpe_ratio = ann_excess_ret / ann_vol if ann_vol != 0 else 0
    
    # 4. Calculate Maximum Drawdown
    # Create a cumulative wealth index starting at 1.0
    cum_wealth = (1 + eval_df_mod[col]).cumprod()
    # Track the highest peak achieved so far
    running_max = cum_wealth.cummax()
    # Calculate the percentage drop from the peak
    drawdowns = (cum_wealth - running_max) / running_max
    max_drawdown = drawdowns.min()
    
    # Save the metrics
    risk_results.append({
        'Strategy': col,
        'Ann_Return': ann_ret,
        'Ann_Volatility': ann_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown
    })

# Convert to a DataFrame for clean formatting
risk_df = pd.DataFrame(risk_results)

# Format the output beautifully for your thesis
risk_df['Ann_Return'] = risk_df['Ann_Return'].map('{:.2%}'.format)
risk_df['Ann_Volatility'] = risk_df['Ann_Volatility'].map('{:.2%}'.format)
risk_df['Sharpe_Ratio'] = risk_df['Sharpe_Ratio'].map('{:.2f}'.format)
risk_df['Max_Drawdown'] = risk_df['Max_Drawdown'].map('{:.2%}'.format)

print("=== RISK-ADJUSTED PERFORMANCE METRICS ===")
print(risk_df)

Calculating Risk Metrics...

=== RISK-ADJUSTED PERFORMANCE METRICS ===
                Strategy Ann_Return Ann_Volatility Sharpe_Ratio Max_Drawdown
0       Net_Ret_LongOnly     30.20%         29.49%         0.91       -9.37%
1      Net_Ret_LongShort     51.60%         31.43%         1.53      -19.07%
2  Net_Ret_Modern_XGB_LS     41.77%         27.92%         1.37      -19.13%
3         Net_Ret_Mom_LS     47.53%         25.73%         1.71       -7.32%
4                     VW      8.41%         14.31%         0.35      -12.75%


In [23]:
# ==========================================
# CELL 3: THE ULTIMATE ALPHA REGRESSIONS
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Running CAPM, FF3, and Carhart 4-Factor Regressions...\n")

# We don't need to run this on the VW market itself, just our strategies
strategies_to_test = [
    'Net_Ret_LongOnly', 
    'Net_Ret_LongShort', 
    'Net_Ret_Modern_XGB_LS', 
    'Net_Ret_Mom_LS'
]

# Create an empty list to store our rows of data
alpha_results = []

for strat in strategies_to_test:
    # Our Dependent Variable (Y) is the Strategy's Excess Return
    Y = eval_df_mod[strat] - eval_df_mod['Rf(1m)']
    
    # ----------------------------------------------------
    # Model 1: CAPM (1-Factor)
    # ----------------------------------------------------
    X_capm = sm.add_constant(eval_df_mod[['Mkt-RF']])
    capm_model = sm.OLS(Y, X_capm).fit()
    
    # ----------------------------------------------------
    # Model 2: Fama-French 3-Factor (FF3)
    # ----------------------------------------------------
    X_ff3 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML']])
    ff3_model = sm.OLS(Y, X_ff3).fit()
    
    # ----------------------------------------------------
    # Model 3: Carhart 4-Factor (FF4)
    # ----------------------------------------------------
    X_ff4 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # Extract Alpha (Intercept) and annualize it (* 12)
    # We also extract the t-stat, p-value, and R-squared
    alpha_results.append({
        'Strategy': strat,
        'CAPM_Alpha': capm_model.params['const'] * 12,
        'CAPM_t': capm_model.tvalues['const'],
        'CAPM_p': capm_model.pvalues['const'],
        
        'FF3_Alpha': ff3_model.params['const'] * 12,
        'FF3_t': ff3_model.tvalues['const'],
        'FF3_p': ff3_model.pvalues['const'],
        
        'FF4_Alpha': ff4_model.params['const'] * 12,
        'FF4_t': ff4_model.tvalues['const'],
        'FF4_p': ff4_model.pvalues['const']
    })

# Format the results into a beautiful Pandas DataFrame
alpha_df = pd.DataFrame(alpha_results)

# Create a mapping function to add significance stars based on p-values
def format_alpha(alpha, pval):
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    return f"{alpha:.2%}{stars}"

# Apply formatting
for model in ['CAPM', 'FF3', 'FF4']:
    alpha_df[f'{model}_Alpha_Str'] = alpha_df.apply(lambda row: format_alpha(row[f'{model}_Alpha'], row[f'{model}_p']), axis=1)
    alpha_df[f'{model}_t'] = alpha_df[f'{model}_t'].map('{:.2f}'.format)
    alpha_df[f'{model}_p'] = alpha_df[f'{model}_p'].map('{:.3f}'.format)

# Reorder columns to match your requested table format perfectly
final_table = alpha_df[[
    'Strategy', 
    'CAPM_Alpha_Str', 'FF3_Alpha_Str', 'FF4_Alpha_Str',
    'CAPM_t', 'FF3_t', 'FF4_t',
    'CAPM_p', 'FF3_p', 'FF4_p'
]]

# Rename for display
final_table.columns = [
    'Strategy', 
    'CAPM Alpha', 'FF3 Alpha', 'FF4 Alpha', 
    'CAPM t-stat', 'FF3 t-stat', 'FF4 t-stat', 
    'CAPM p-val', 'FF3 p-val', 'FF4 p-val'
]

print("=== COMBINED ALPHA SUMMARY TABLE ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
display(final_table)

Running CAPM, FF3, and Carhart 4-Factor Regressions...

=== COMBINED ALPHA SUMMARY TABLE ===
Note: *** p<0.01, ** p<0.05, * p<0.10



,Strategy,CAPM Alpha,FF3 Alpha,FF4 Alpha,CAPM t-stat,FF3 t-stat,FF4 t-stat,CAPM p-val,FF3 p-val,FF4 p-val
0,Net_Ret_LongOnly,28.28%,31.75%**,33.88%**,1.63,2.23,2.34,0.113,0.033,0.026
1,Net_Ret_LongShort,48.47%**,58.35%***,57.76%***,2.59,3.81,3.66,0.014,0.001,0.001
2,Net_Ret_Modern_XGB_LS,38.73%**,44.71%***,44.85%***,2.33,3.26,3.17,0.026,0.003,0.003
3,Net_Ret_Mom_LS,45.32%***,50.24%***,50.00%***,2.98,3.87,3.74,0.005,0.001,0.001


In [24]:
# ==========================================
# CELL 4: FF4 FACTOR LOADINGS (THE "BLACK BOX" EXPLAINER)
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Extracting Detailed Carhart 4-Factor Loadings...\n")

# Let's look closely at the Mutual Fund (Long-Only) and the Hedge Fund (Long-Short)
strategies_to_analyze = {
    'Modern Ensemble (Long-Short)': 'Net_Ret_LongShort',
    'Modern Ensemble (Long-Only)': 'Net_Ret_LongOnly'
}

for name, col in strategies_to_analyze.items():
    print(f"=== {name} FF4 Factor Loadings ===")
    
    # 1. Setup the Regression exactly like Cell 3
    Y = eval_df_mod[col] - eval_df_mod['Rf(1m)']
    X_ff4 = sm.add_constant(eval_df_mod[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # 2. Extract the detailed statistics
    loadings_df = pd.DataFrame({
        'Coef': ff4_model.params,
        't-stat': ff4_model.tvalues,
        'p-value': ff4_model.pvalues
    })
    
    # 3. Add significance stars
    def get_stars(pval):
        if pval < 0.01: return '***'
        elif pval < 0.05: return '**'
        elif pval < 0.10: return '*'
        return ''
    
    loadings_df['sig'] = loadings_df['p-value'].apply(get_stars)
    
    # 4. Clean up the row names for display
    loadings_df.index = ['Alpha (monthly)', 'Market (β_mkt)', 'SMB (β_smb)', 'HML (β_hml)', 'UMD (β_umd)']
    
    # Format the numbers
    loadings_df['Coef'] = loadings_df['Coef'].map('{:.6f}'.format)
    loadings_df['t-stat'] = loadings_df['t-stat'].map('{:.6f}'.format)
    loadings_df['p-value'] = loadings_df['p-value'].map('{:.6e}'.format)
    
    print(loadings_df)
    print("\n")

Extracting Detailed Carhart 4-Factor Loadings...

=== Modern Ensemble (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.048131   3.661211  9.598271e-04  ***
Market (β_mkt)    0.330383   1.094989  2.822349e-01     
SMB (β_smb)      -0.413282  -1.598246  1.204691e-01     
HML (β_hml)      -0.563477  -2.648970  1.275289e-02   **
UMD (β_umd)       0.071878   0.232510  8.177211e-01     


=== Modern Ensemble (Long-Only) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.028232   2.344612  2.585841e-02   **
Market (β_mkt)    0.023578   0.085314  9.325783e-01     
SMB (β_smb)      -0.694180  -2.930850  6.408866e-03  ***
HML (β_hml)      -0.264195  -1.355969  1.852284e-01     
UMD (β_umd)      -0.258891  -0.914291  3.678544e-01     




### Historic evaluation

In [25]:
# ==========================================
# CELL 2: RISK METRICS (SHARPE & DRAWDOWN)
# ==========================================
import pandas as pd
import numpy as np

print("Calculating Risk Metrics...\n")

risk_results = []

for col in strategy_cols_hist:
    # 1. Calculate Excess Return for the strategy
    # Note: If the strategy is 'VW', it is already the market, but we still subtract Rf for its Sharpe
    excess_return = eval_df_hist[col] - eval_df_hist['Rf(1m)']
    
    # 2. Calculate Annualized Return and Volatility
    ann_ret = eval_df_hist[col].mean() * 12
    ann_vol = eval_df_hist[col].std() * np.sqrt(12)
    
    # 3. Calculate Annualized Sharpe Ratio
    ann_excess_ret = excess_return.mean() * 12
    sharpe_ratio = ann_excess_ret / ann_vol if ann_vol != 0 else 0
    
    # 4. Calculate Maximum Drawdown
    # Create a cumulative wealth index starting at 1.0
    cum_wealth = (1 + eval_df_hist[col]).cumprod()
    # Track the highest peak achieved so far
    running_max = cum_wealth.cummax()
    # Calculate the percentage drop from the peak
    drawdowns = (cum_wealth - running_max) / running_max
    max_drawdown = drawdowns.min()
    
    # Save the metrics
    risk_results.append({
        'Strategy': col,
        'Ann_Return': ann_ret,
        'Ann_Volatility': ann_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown': max_drawdown
    })

# Convert to a DataFrame for clean formatting
risk_df = pd.DataFrame(risk_results)

# Format the output beautifully for your thesis
risk_df['Ann_Return'] = risk_df['Ann_Return'].map('{:.2%}'.format)
risk_df['Ann_Volatility'] = risk_df['Ann_Volatility'].map('{:.2%}'.format)
risk_df['Sharpe_Ratio'] = risk_df['Sharpe_Ratio'].map('{:.2f}'.format)
risk_df['Max_Drawdown'] = risk_df['Max_Drawdown'].map('{:.2%}'.format)

print("=== RISK-ADJUSTED PERFORMANCE METRICS ===")
print(risk_df)

Calculating Risk Metrics...

=== RISK-ADJUSTED PERFORMANCE METRICS ===
              Strategy Ann_Return Ann_Volatility Sharpe_Ratio Max_Drawdown
0     Net_Ret_LongOnly     32.07%         28.76%         1.00       -8.80%
1    Net_Ret_LongShort     57.07%         29.87%         1.80      -18.35%
2  Net_Ret_Hist_XGB_LS     56.79%         30.11%         1.77      -16.77%
3       Net_Ret_Mom_LS     47.53%         25.73%         1.71       -7.32%
4                   VW      8.41%         14.31%         0.35      -12.75%


In [26]:
# ==========================================
# CELL 3: THE ULTIMATE ALPHA REGRESSIONS
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Running CAPM, FF3, and Carhart 4-Factor Regressions...\n")

# We don't need to run this on the VW market itself, just our strategies
strategies_to_test = [
    'Net_Ret_LongOnly', 
    'Net_Ret_LongShort', 
    'Net_Ret_Hist_XGB_LS', 
    'Net_Ret_Mom_LS'
]

# Create an empty list to store our rows of data
alpha_results = []

for strat in strategies_to_test:
    # Our Dependent Variable (Y) is the Strategy's Excess Return
    Y = eval_df_hist[strat] - eval_df_hist['Rf(1m)']
    
    # ----------------------------------------------------
    # Model 1: CAPM (1-Factor)
    # ----------------------------------------------------
    X_capm = sm.add_constant(eval_df_hist[['Mkt-RF']])
    capm_model = sm.OLS(Y, X_capm).fit()
    
    # ----------------------------------------------------
    # Model 2: Fama-French 3-Factor (FF3)
    # ----------------------------------------------------
    X_ff3 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML']])
    ff3_model = sm.OLS(Y, X_ff3).fit()
    
    # ----------------------------------------------------
    # Model 3: Carhart 4-Factor (FF4)
    # ----------------------------------------------------
    X_ff4 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # Extract Alpha (Intercept) and annualize it (* 12)
    # We also extract the t-stat, p-value, and R-squared
    alpha_results.append({
        'Strategy': strat,
        'CAPM_Alpha': capm_model.params['const'] * 12,
        'CAPM_t': capm_model.tvalues['const'],
        'CAPM_p': capm_model.pvalues['const'],
        
        'FF3_Alpha': ff3_model.params['const'] * 12,
        'FF3_t': ff3_model.tvalues['const'],
        'FF3_p': ff3_model.pvalues['const'],
        
        'FF4_Alpha': ff4_model.params['const'] * 12,
        'FF4_t': ff4_model.tvalues['const'],
        'FF4_p': ff4_model.pvalues['const']
    })

# Format the results into a beautiful Pandas DataFrame
alpha_df = pd.DataFrame(alpha_results)

# Create a mapping function to add significance stars based on p-values
def format_alpha(alpha, pval):
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    return f"{alpha:.2%}{stars}"

# Apply formatting
for model in ['CAPM', 'FF3', 'FF4']:
    alpha_df[f'{model}_Alpha_Str'] = alpha_df.apply(lambda row: format_alpha(row[f'{model}_Alpha'], row[f'{model}_p']), axis=1)
    alpha_df[f'{model}_t'] = alpha_df[f'{model}_t'].map('{:.2f}'.format)
    alpha_df[f'{model}_p'] = alpha_df[f'{model}_p'].map('{:.3f}'.format)

# Reorder columns to match your requested table format perfectly
final_table = alpha_df[[
    'Strategy', 
    'CAPM_Alpha_Str', 'FF3_Alpha_Str', 'FF4_Alpha_Str',
    'CAPM_t', 'FF3_t', 'FF4_t',
    'CAPM_p', 'FF3_p', 'FF4_p'
]]

# Rename for display
final_table.columns = [
    'Strategy', 
    'CAPM Alpha', 'FF3 Alpha', 'FF4 Alpha', 
    'CAPM t-stat', 'FF3 t-stat', 'FF4 t-stat', 
    'CAPM p-val', 'FF3 p-val', 'FF4 p-val'
]

print("=== COMBINED ALPHA SUMMARY TABLE ===")
print("Note: *** p<0.01, ** p<0.05, * p<0.10\n")
display(final_table)

Running CAPM, FF3, and Carhart 4-Factor Regressions...

=== COMBINED ALPHA SUMMARY TABLE ===
Note: *** p<0.01, ** p<0.05, * p<0.10



,Strategy,CAPM Alpha,FF3 Alpha,FF4 Alpha,CAPM t-stat,FF3 t-stat,FF4 t-stat,CAPM p-val,FF3 p-val,FF4 p-val
0,Net_Ret_LongOnly,30.11%*,28.78%*,31.02%**,1.77,1.97,2.09,0.085,0.058,0.045
1,Net_Ret_LongShort,54.30%***,59.29%***,58.63%***,3.05,3.86,3.71,0.004,0.001,0.001
2,Net_Ret_Hist_XGB_LS,53.75%***,59.28%***,59.07%***,2.99,4.12,3.99,0.005,0.000,0.000
3,Net_Ret_Mom_LS,45.32%***,50.24%***,50.00%***,2.98,3.87,3.74,0.005,0.001,0.001


In [27]:
# ==========================================
# CELL 4: FF4 FACTOR LOADINGS (THE "BLACK BOX" EXPLAINER)
# ==========================================
import statsmodels.api as sm
import pandas as pd

print("Extracting Detailed Carhart 4-Factor Loadings...\n")

# Let's look closely at the Mutual Fund (Long-Only) and the Hedge Fund (Long-Short)
strategies_to_analyze = {
    'Historical Ensemble (Long-Short)': 'Net_Ret_LongShort',
    'Historical Ensemble (Long-Only)': 'Net_Ret_LongOnly',
    'Historical XGB (Long-Short)': 'Net_Ret_Hist_XGB_LS',
}

for name, col in strategies_to_analyze.items():
    print(f"=== {name} FF4 Factor Loadings ===")
    
    # 1. Setup the Regression exactly like Cell 3
    Y = eval_df_hist[col] - eval_df_hist['Rf(1m)']
    X_ff4 = sm.add_constant(eval_df_hist[['Mkt-RF', 'SMB', 'HML', 'UMD']])
    ff4_model = sm.OLS(Y, X_ff4).fit()
    
    # 2. Extract the detailed statistics
    loadings_df = pd.DataFrame({
        'Coef': ff4_model.params,
        't-stat': ff4_model.tvalues,
        'p-value': ff4_model.pvalues
    })
    
    # 3. Add significance stars
    def get_stars(pval):
        if pval < 0.01: return '***'
        elif pval < 0.05: return '**'
        elif pval < 0.10: return '*'
        return ''
    
    loadings_df['sig'] = loadings_df['p-value'].apply(get_stars)
    
    # 4. Clean up the row names for display
    loadings_df.index = ['Alpha (monthly)', 'Market (β_mkt)', 'SMB (β_smb)', 'HML (β_hml)', 'UMD (β_umd)']
    
    # Format the numbers
    loadings_df['Coef'] = loadings_df['Coef'].map('{:.6f}'.format)
    loadings_df['t-stat'] = loadings_df['t-stat'].map('{:.6f}'.format)
    loadings_df['p-value'] = loadings_df['p-value'].map('{:.6e}'.format)
    
    print(loadings_df)
    print("\n")

Extracting Detailed Carhart 4-Factor Loadings...

=== Historical Ensemble (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.048862   3.712157  8.366940e-04  ***
Market (β_mkt)    0.203170   0.672522  5.063990e-01     
SMB (β_smb)      -0.501853  -1.938332  6.204300e-02    *
HML (β_hml)      -0.395119  -1.855166  7.342328e-02    *
UMD (β_umd)       0.079106   0.255568  8.000287e-01     


=== Historical Ensemble (Long-Only) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.025848   2.093517  4.485837e-02   **
Market (β_mkt)   -0.021317  -0.075229  9.405318e-01     
SMB (β_smb)      -0.788528  -3.246928  2.868774e-03  ***
HML (β_hml)      -0.094092  -0.470988  6.410560e-01     
UMD (β_umd)      -0.271710  -0.935853  3.568219e-01     


=== Historical XGB (Long-Short) FF4 Factor Loadings ===
                      Coef     t-stat       p-value  sig
Alpha (monthly)   0.049223  